In [ ]:
import numpy
import numpy
import jax.numpy as jnp
import numpy as np
# import seaborn as sns
import pandas as pd
from tqdm.autonotebook import tqdm
import matplotlib.pyplot as plt
import copy
from jaxopt import ScipyBoundedMinimize, LBFGS

In [ ]:
rng = numpy.random.default_rng(0)
should_log = True

In [ ]:
from adaptive_latents.sim_stim import make_sr
from adaptive_latents import datasets
from adaptive_latents.stim_designer import OptimizationMethod
import seaborn as sns
from adaptive_latents.utils import angle_between

In [ ]:
from adaptive_latents.regressions import _OutmodedBaseMultiKernelRegressor, BaseMultiKernelRegressor

logs = {}

sr, stim_designer, log = make_sr(
    input_array=datasets.Odoherty21Dataset().neural_data,
    rng=copy.deepcopy(rng),
    optimization_method=OptimizationMethod.JAXOPT,
    # stim_direction_type='random_feasible',
    stim_direction_type='first',
    show_tqdm=True,
    u_to_s_model_type='kernel_regressed',
    exit_time=60,
    stim_reg=_OutmodedBaseMultiKernelRegressor(length_scales=[0.04, 0.04, 0.04], maxlen=500),
    lam_1=0.001,
)
logs['a'] = stim_designer.log


sr, stim_designer, log = make_sr(
    input_array=datasets.Odoherty21Dataset().neural_data,
    rng=copy.deepcopy(rng),
    optimization_method=OptimizationMethod.JAXOPT,
    # stim_direction_type='random_feasible',
    stim_direction_type='first',
    show_tqdm=True,
    u_to_s_model_type='kernel_regressed',
    exit_time=60,
    stim_reg=BaseMultiKernelRegressor(length_scales=[0.04, 0.04, 0.04], maxlen=500),
    lam_1=0.001,
)
logs['b'] = stim_designer.log



In [ ]:
logs['a'][-1].keys()

In [ ]:
def f(d):
    u = d['u'].copy()
    # return d['optimization_time']
    # return angle_between(d['observed_s_hat'], d['v'])
    return angle_between(d['s'], d['v'])
    # return len(d['intermediate_xs']) if 'intermediate_xs' in d else np.nan
    # return np.linalg.norm(d['observed_s_hat'])


data = {k: np.array([f(d) for d in v]) for k, v in logs.items()}

fig, ax = plt.subplots()

plt.plot(np.vstack([data['a'], data['b']]), '.-k', alpha=0.5)
ax.set_xticks([0,1])
ax.set_xticklabels(['a', 'b'])
ax.set_ylabel('angle between s and v (degrees)')
ax.set_xlabel('design method');


In [ ]:
[len(l['intermediate_xs']) for l in  logs['b'] if 'intermediate_xs' in l]

In [ ]:
%matplotlib inline
fig, ax = plt.subplots()

for i in range(len(logs['b'])):
    l_a, l_b = logs['a'][i], logs['b'][i]
    if 'intermediate_objective' not in l_a:
        continue


    blue_line = ax.plot(l_a['intermediate_objective'], '.-C0')
    if not l_a['result'].state.success:
        blue_x = ax.plot(len(l_a['intermediate_objective'])-1, l_a['intermediate_objective'][-1], 'xC0', markersize=10)

    orange_line = ax.plot(l_b['intermediate_objective'], '.-C1')
    if not l_b['result'].state.success:
        orange_x = ax.plot(len(l_b['intermediate_objective'])-1, l_b['intermediate_objective'][-1], 'xC1', markersize=10)

ax.legend([blue_line[0], orange_line[0], blue_x[0]], ['old kernel regression', 'new kernel regression', 'failed'])
ax.set_xlabel('iteration')
ax.set_ylabel('objective value');

In [ ]:
fig, axs = plt.subplots(nrows=3, sharex=True)

l_b = logs['b'][25]

axs[0].plot(l_b['intermediate_objective'], '.-C0', label='total objective')
axs[1].plot(l_b['intermediate_objective_l1'], '.-C1', label='L1 objective')
axs[2].plot(l_b['intermediate_objective_proj'], '.-C2', label='projection objective')
for ax in axs:
    ax.legend()
